# House Prices: BaseMLP with Id=1299 Outlier Exclusion

## tl;dr

- 원본: 실행 검증된 `notebooks/house_prices_basemlp.ipynb`
- 유일한 정책 변경: 각 fold 학습에서 **`Id=1299`만 제외**, validation 1,460행 유지
- 5-fold RMSLE: **0.150128 ± 0.041293**
- OOF RMSLE: **0.154604**
- 이전 `1299` 민감도 실행과 checkpoint 가중치 **5/5 정확히 일치**
- test **1,459행** submission 생성 및 형식 검증 완료
- Kaggle public/private score: **unverified**

## Context & Methods

이 노트북은 검증된 `notebooks/house_prices_basemlp.ipynb`를 직접 복제하고, `Id=1299`만 각 fold의 학습 인덱스에서 제외한 제출용 실험이다. 모델 구조, 전처리, fold, seed, optimizer, loss, early stopping, test 추론은 원본 BaseMLP와 동일하다.

### Key Assumptions

- RMSLE는 `log1p(SalePrice)` 공간의 RMSE와 동일하게 계산한다.
- `Id=1299`는 fold 학습에서만 제외하고 validation 1,460행은 모두 OOF에 유지한다.
- raw train/test 파일과 원본 BaseMLP 노트북은 수정하지 않는다.
- test는 스키마·Id 순서 검증과 blind inference 외에는 검사하거나 해석하지 않는다.
- 모델 클래스와 학습 루프는 원본 노트북과 동일하게 canonical `scripts/run_basemlp.py`에서 import한다.

In [1]:
from __future__ import annotations

import hashlib
import inspect
import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import KFold

ROOT = Path.cwd()
if not (ROOT / "data" / "train.csv").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT.resolve()))

import scripts.run_basemlp as impl

TRAIN_PATH = ROOT / "data" / "train.csv"
TEST_PATH = ROOT / "data" / "test.csv"
SAMPLE_SUBMISSION_PATH = ROOT / "data" / "sample_submission.csv"
SOURCE_NOTEBOOK_PATH = ROOT / "notebooks" / "house_prices_basemlp.ipynb"
REFERENCE_EXPERIMENT_ID = "BASEMLP-20260719-2H-01"
SENSITIVITY_EXPERIMENT_ID = "BASEMLP-20260719-OUTLIER-1299"
NOTEBOOK_EXPERIMENT_ID = "BASEMLP-20260719-OUTLIER-1299-NOTEBOOK-SUB-01"
EXCLUDED_TRAINING_IDS = (1299,)
REFERENCE_METRICS_PATH = ROOT / "reports" / f"basemlp_metrics_{REFERENCE_EXPERIMENT_ID}.json"
SENSITIVITY_METRICS_PATH = ROOT / "reports" / f"basemlp_outlier_metrics_{SENSITIVITY_EXPERIMENT_ID}.json"
NOTEBOOK_SUBMISSION_PATH = ROOT / "submissions" / f"submission_{NOTEBOOK_EXPERIMENT_ID}.csv"
NOTEBOOK_OOF_PATH = ROOT / "reports" / f"basemlp_outlier_1299_submission_oof_{NOTEBOOK_EXPERIMENT_ID}.csv"
NOTEBOOK_FOLD_PREDICTIONS_PATH = ROOT / "reports" / f"basemlp_outlier_1299_submission_test_predictions_{NOTEBOOK_EXPERIMENT_ID}.csv"
METRICS_PATH = ROOT / "reports" / f"basemlp_outlier_1299_submission_metrics_{NOTEBOOK_EXPERIMENT_ID}.json"
SUBMISSION_VALIDATION_PATH = ROOT / "reports" / f"basemlp_outlier_1299_submission_validation_{NOTEBOOK_EXPERIMENT_ID}.json"

train = pd.read_csv(TRAIN_PATH, keep_default_na=False)
test = pd.read_csv(TEST_PATH, keep_default_na=False)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
categorical_columns = [
    column
    for column in train.columns
    if column not in {*impl.NUMERIC_COLUMNS, "Id", "SalePrice"}
]

assert train.shape == (1460, 81)
assert test.shape == (1459, 80)
assert list(test.columns) == [column for column in train.columns if column != "SalePrice"]
assert list(sample_submission.columns) == ["Id", "SalePrice"]
assert sample_submission["Id"].equals(test["Id"])
assert train["Id"].is_unique and test["Id"].is_unique
assert set(EXCLUDED_TRAINING_IDS).issubset(set(train["Id"]))
assert REFERENCE_METRICS_PATH.exists() and SENSITIVITY_METRICS_PATH.exists()
print(f"train={train.shape}, test={test.shape}, categorical={len(categorical_columns)}")

train=(1460, 81), test=(1459, 80), categorical=45


## Model & Preprocessing

원본 BaseMLP와 동일하게 수치형은 fold median imputation과 표준화, 범주형은 구조적 부재 라벨과 fold one-hot encoding을 사용한다. 타깃은 fold 학습 부분의 평균·표준편차로 표준화한 `log1p(SalePrice)`다. 유일한 학습 정책 변경은 `Id=1299` 제외다.

In [2]:
print(inspect.getsource(impl.BaseMLP))
display(pd.Series({
    "hidden_dims": impl.HIDDEN_DIMS,
    "activation": "ReLU",
    "dropout": 0.0,
    "batch_normalization": False,
    "optimizer": "Adam",
    "learning_rate": impl.LEARNING_RATE,
    "weight_decay": impl.WEIGHT_DECAY,
    "loss": "MSELoss",
    "batch_size": impl.BATCH_SIZE,
    "max_epochs": impl.MAX_EPOCHS,
    "patience": impl.PATIENCE,
    "device": str(impl.DEVICE),
}, name="BaseMLP configuration"))

class BaseMLP(nn.Module):
    """Small tabular MLP used as the fixed neural-network reference."""

    def __init__(self, input_dim: int) -> None:
        super().__init__()
        self.hidden1 = nn.Linear(input_dim, HIDDEN_DIMS[0])
        self.hidden2 = nn.Linear(HIDDEN_DIMS[0], HIDDEN_DIMS[1])
        self.output = nn.Linear(HIDDEN_DIMS[1], 1)
        self.activation = nn.ReLU()
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.kaiming_normal_(self.hidden1.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden1.bias)
        nn.init.kaiming_normal_(self.hidden2.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden2.bias)
        nn.init.xavier_normal_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        hidden = self.activation(self.hidden1(inputs))
        hidden = self.activation(self.hidden2(hidden))
        return self.output(hidden).squeeze(1)

hidden_dims            (128, 64)
activation                  ReLU
dropout                      0.0
batch_normalization        False
optimizer                   Adam
learning_rate              0.001
weight_decay                 0.0
loss                     MSELoss
batch_size                    64
max_epochs                   200
patience                      25
device                       cpu
Name: BaseMLP configuration, dtype: object

## Data

학습 입력과 target을 준비한다. test는 같은 row-only cleaning을 적용한 뒤 fold별 blind inference에만 전달한다.

In [3]:
X = impl.clean_rows(train.drop(columns="SalePrice"), categorical_columns)
X_test = impl.clean_rows(test, categorical_columns)
y = train["SalePrice"].to_numpy(dtype=np.float64)
y_log = np.log1p(y)
excluded_mask = train["Id"].isin(EXCLUDED_TRAINING_IDS).to_numpy()

checkpoint_dir = ROOT / "artifacts" / "checkpoints" / NOTEBOOK_EXPERIMENT_ID
preprocessor_dir = ROOT / "artifacts" / "preprocessors" / NOTEBOOK_EXPERIMENT_ID
epoch_log_dir = ROOT / "artifacts" / "logs" / NOTEBOOK_EXPERIMENT_ID
for directory in (checkpoint_dir, preprocessor_dir, epoch_log_dir):
    directory.mkdir(parents=True, exist_ok=True)
NOTEBOOK_SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"model inputs={X.shape}, target={y_log.shape}")
print(f"fold-training exclusion={EXCLUDED_TRAINING_IDS}; validation exclusion=none")

model inputs=(1460, 80), target=(1460,)
fold-training exclusion=(1299,); validation exclusion=none


## Results

원본 BaseMLP와 같은 folds에서 `Id=1299`만 각 fold의 후보 학습 인덱스에서 제거한다. 전처리기와 모델은 fold마다 새로 적합하고, 저장한 best checkpoint와 전처리기를 다시 불러 OOF와 test를 추론한다.

In [4]:
started_time = time.perf_counter()
kfold = KFold(n_splits=impl.N_SPLITS, shuffle=True, random_state=impl.SEED)
oof_log_prediction = np.full(len(X), np.nan, dtype=np.float64)
test_log_predictions = []
fold_rows = []

for fold, (candidate_train_index, valid_index) in enumerate(kfold.split(X), start=1):
    train_index = candidate_train_index[~excluded_mask[candidate_train_index]]
    fold_seed = impl.SEED + fold
    checkpoint_path = checkpoint_dir / f"fold_{fold}_best.pt"
    preprocessor_path = preprocessor_dir / f"fold_{fold}_preprocessor.joblib"
    epoch_log_path = epoch_log_dir / f"fold_{fold}_epochs.csv"

    preprocessor = impl.make_preprocessor(categorical_columns)
    X_train = preprocessor.fit_transform(X.iloc[train_index]).astype(np.float32)
    X_valid = preprocessor.transform(X.iloc[valid_index]).astype(np.float32)
    joblib.dump(preprocessor, preprocessor_path)

    _, training = impl.train_fold(
        X_train,
        y_log[train_index],
        X_valid,
        y_log[valid_index],
        seed=fold_seed,
        experiment_id=NOTEBOOK_EXPERIMENT_ID,
        fold=fold,
        checkpoint_path=checkpoint_path,
        epoch_log_path=epoch_log_path,
    )

    saved_preprocessor = joblib.load(preprocessor_path)
    X_valid_saved = saved_preprocessor.transform(X.iloc[valid_index]).astype(np.float32)
    X_test_saved = saved_preprocessor.transform(X_test).astype(np.float32)
    checkpoint = torch.load(checkpoint_path, map_location=impl.DEVICE, weights_only=False)
    restored_model = impl.BaseMLP(checkpoint["input_dim"]).to(impl.DEVICE)
    restored_model.load_state_dict(checkpoint["model_state_dict"])

    valid_prediction = impl.predict_log_target(
        restored_model, X_valid_saved, checkpoint["target_mean"], checkpoint["target_std"]
    )
    test_prediction = impl.predict_log_target(
        restored_model, X_test_saved, checkpoint["target_mean"], checkpoint["target_std"]
    )
    oof_log_prediction[valid_index] = valid_prediction
    test_log_predictions.append(test_prediction)

    fold_rmsle = float(np.sqrt(np.mean((y_log[valid_index] - valid_prediction) ** 2)))
    excluded_from_training = sorted(
        set(train.iloc[candidate_train_index]["Id"].astype(int))
        & set(EXCLUDED_TRAINING_IDS)
    )
    excluded_in_validation = sorted(
        set(train.iloc[valid_index]["Id"].astype(int))
        & set(EXCLUDED_TRAINING_IDS)
    )
    fold_rows.append({
        "fold": fold,
        "seed": fold_seed,
        "candidate_train_rows": len(candidate_train_index),
        "effective_train_rows": len(train_index),
        "validation_rows": len(valid_index),
        "excluded_from_training": excluded_from_training,
        "excluded_in_validation_but_retained": excluded_in_validation,
        "encoded_features": X_train.shape[1],
        "best_epoch": training["best_epoch"],
        "stopping_epoch": training["stopping_epoch"],
        "rmsle": fold_rmsle,
    })
    print(
        f"fold {fold}: RMSLE={fold_rmsle:.6f}, train_rows={len(train_index)}, "
        f"best={training['best_epoch']}, stop={training['stopping_epoch']}"
    )

assert not np.isnan(oof_log_prediction).any()
fold_results = pd.DataFrame(fold_rows)
display(fold_results)

fold 1: RMSLE=0.138678, train_rows=1167, best=5, stop=30


fold 2: RMSLE=0.128827, train_rows=1167, best=6, stop=31


fold 3: RMSLE=0.222199, train_rows=1168, best=5, stop=30


fold 4: RMSLE=0.142046, train_rows=1167, best=5, stop=30


fold 5: RMSLE=0.118889, train_rows=1167, best=7, stop=32


,fold,seed,candidate_train_rows,effective_train_rows,validation_rows,excluded_from_training,excluded_in_validation_but_retained,encoded_features,best_epoch,stopping_epoch,rmsle
0,1,43,1168,1167,292,[1299],[],326,5,30,0.138678
1,2,44,1168,1167,292,[1299],[],326,6,31,0.128827
2,3,45,1168,1168,292,[],[1299],323,5,30,0.222199
3,4,46,1168,1167,292,[1299],[],323,5,30,0.142046
4,5,47,1168,1167,292,[1299],[],327,7,32,0.118889


In [5]:
fold_scores = fold_results["rmsle"].to_numpy(dtype=np.float64)
cv_mean = float(fold_scores.mean())
cv_std = float(fold_scores.std(ddof=1))
oof_price_prediction = np.clip(np.expm1(oof_log_prediction), 0, None)
oof_rmsle = float(np.sqrt(np.mean((y_log - np.log1p(oof_price_prediction)) ** 2)))

test_log_prediction_matrix = np.vstack(test_log_predictions)
mean_test_log_prediction = test_log_prediction_matrix.mean(axis=0)
test_price_prediction = np.clip(np.expm1(mean_test_log_prediction), 0, None)

pd.DataFrame({
    "Id": train["Id"],
    "SalePrice": y,
    "oof_log_prediction": oof_log_prediction,
    "oof_saleprice_prediction": oof_price_prediction,
}).to_csv(NOTEBOOK_OOF_PATH, index=False)

pd.DataFrame({
    "Id": test["Id"],
    **{
        f"fold_{fold}_log_prediction": test_log_prediction_matrix[fold - 1]
        for fold in range(1, impl.N_SPLITS + 1)
    },
    "mean_log_prediction": mean_test_log_prediction,
    "SalePrice": test_price_prediction,
}).to_csv(NOTEBOOK_FOLD_PREDICTIONS_PATH, index=False)

notebook_submission = sample_submission.copy()
notebook_submission["SalePrice"] = test_price_prediction
notebook_submission.to_csv(NOTEBOOK_SUBMISSION_PATH, index=False)

display(pd.Series({
    "cv_mean": cv_mean,
    "cv_std": cv_std,
    "oof_rmsle": oof_rmsle,
    "submission_rows": len(notebook_submission),
}, name="Notebook reproduction results"))

cv_mean               0.150128
cv_std                0.041293
oof_rmsle             0.154604
submission_rows    1459.000000
Name: Notebook reproduction results, dtype: float64

## Reproducibility & Submission Checks

저장한 submission을 다시 읽고 원본 BaseMLP의 스키마와 비교한다. fold/OOF 결과와 checkpoint는 앞서 독립 실행한 `1299`-only 민감도 실험과 대조한다. 개별 test 예측은 표시하거나 해석하지 않는다.

In [6]:
def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

reloaded_submission = pd.read_csv(NOTEBOOK_SUBMISSION_PATH)
reference_metrics = json.loads(REFERENCE_METRICS_PATH.read_text(encoding="utf-8"))
sensitivity_metrics = json.loads(SENSITIVITY_METRICS_PATH.read_text(encoding="utf-8"))
sensitivity_checkpoint_dir = ROOT / sensitivity_metrics["artifacts"]["checkpoint_dir"]

checkpoint_weight_checks = []
for fold in range(1, impl.N_SPLITS + 1):
    notebook_checkpoint = torch.load(
        checkpoint_dir / f"fold_{fold}_best.pt",
        map_location="cpu",
        weights_only=False,
    )
    sensitivity_checkpoint = torch.load(
        sensitivity_checkpoint_dir / f"fold_{fold}_best.pt",
        map_location="cpu",
        weights_only=False,
    )
    checkpoint_weight_checks.append(all(
        torch.equal(
            notebook_checkpoint["model_state_dict"][key],
            sensitivity_checkpoint["model_state_dict"][key],
        )
        for key in notebook_checkpoint["model_state_dict"]
    ))

checks = {
    "columns": reloaded_submission.columns.tolist() == ["Id", "SalePrice"],
    "rows": len(reloaded_submission) == len(test) == 1459,
    "dtypes": reloaded_submission.dtypes.astype(str).to_dict() == {"Id": "int64", "SalePrice": "float64"},
    "id_order": reloaded_submission["Id"].equals(test["Id"]),
    "unique_ids": reloaded_submission["Id"].is_unique,
    "missing_predictions": int(reloaded_submission["SalePrice"].isna().sum()) == 0,
    "finite_predictions": bool(np.isfinite(reloaded_submission["SalePrice"]).all()),
    "positive_predictions": bool(reloaded_submission["SalePrice"].gt(0).all()),
    "fold_scores_exact": np.array_equal(
        fold_scores,
        np.asarray(sensitivity_metrics["results"]["fold_scores"], dtype=np.float64),
    ),
    "oof_rmsle_exact": oof_rmsle == float(sensitivity_metrics["results"]["oof_rmsle"]),
    "checkpoint_weights_exact_5_of_5": all(checkpoint_weight_checks),
    "fold_artifacts": all(
        len(list(directory.glob(pattern))) == impl.N_SPLITS
        for directory, pattern in (
            (checkpoint_dir, "fold_*_best.pt"),
            (preprocessor_dir, "fold_*_preprocessor.joblib"),
            (epoch_log_dir, "fold_*_epochs.csv"),
        )
    ),
}
submission_validation = {
    "experiment_id": NOTEBOOK_EXPERIMENT_ID,
    "submission_path": str(NOTEBOOK_SUBMISSION_PATH.relative_to(ROOT)),
    "columns": reloaded_submission.columns.tolist(),
    "rows": int(len(reloaded_submission)),
    "dtypes": reloaded_submission.dtypes.astype(str).to_dict(),
    "id_matches_test_order": bool(reloaded_submission["Id"].equals(test["Id"])),
    "unique_ids": bool(reloaded_submission["Id"].is_unique),
    "missing_predictions": int(reloaded_submission["SalePrice"].isna().sum()),
    "nonfinite_predictions": int((~np.isfinite(reloaded_submission["SalePrice"])).sum()),
    "nonpositive_predictions": int(reloaded_submission["SalePrice"].le(0).sum()),
    "prediction_min": float(reloaded_submission["SalePrice"].min()),
    "prediction_max": float(reloaded_submission["SalePrice"].max()),
    "checkpoint_weight_matches_sensitivity_folds": int(sum(checkpoint_weight_checks)),
    "checks": checks,
    "all_checks_passed": all(checks.values()),
    "kaggle_public_score": "unverified",
    "kaggle_private_score": "unverified",
}
metrics = {
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "experiment_id": NOTEBOOK_EXPERIMENT_ID,
    "source_notebook": str(SOURCE_NOTEBOOK_PATH.relative_to(ROOT)),
    "source_notebook_sha256": sha256(SOURCE_NOTEBOOK_PATH),
    "source": {
        "train_path": "data/train.csv",
        "train_sha256": sha256(TRAIN_PATH),
        "test_path": "data/test.csv",
        "test_sha256": sha256(TEST_PATH),
        "sample_submission_path": "data/sample_submission.csv",
        "sample_submission_sha256": sha256(SAMPLE_SUBMISSION_PATH),
    },
    "intervention": {
        "name": "Outlier fold-training exclusion",
        "excluded_ids": list(EXCLUDED_TRAINING_IDS),
        "scope": "fold training only",
        "validation_rows_retained": 1460,
        "other_recipe_changes_vs_source_notebook": "none",
    },
    "model": {
        "class": "BaseMLP(nn.Module)",
        "architecture": f"input->{list(impl.HIDDEN_DIMS)}->1",
        "activation": "ReLU",
        "optimizer": "Adam",
        "learning_rate": impl.LEARNING_RATE,
        "weight_decay": impl.WEIGHT_DECAY,
        "loss": "MSELoss on fold-standardized log1p(SalePrice)",
        "batch_size": impl.BATCH_SIZE,
        "max_epochs": impl.MAX_EPOCHS,
        "patience": impl.PATIENCE,
        "seed": impl.SEED,
        "fold_seeds": list(range(impl.SEED + 1, impl.SEED + impl.N_SPLITS + 1)),
        "best_epochs": fold_results["best_epoch"].tolist(),
        "stopping_epochs": fold_results["stopping_epoch"].tolist(),
        "device": str(impl.DEVICE),
        "precision": "float32",
    },
    "validation": {
        "splitter": "KFold(n_splits=5,shuffle=True,random_state=42)",
        "metric": "RMSLE / RMSE on log1p target",
        "fold_scores": fold_scores.tolist(),
        "cv_mean": cv_mean,
        "cv_std": cv_std,
        "oof_rmsle": oof_rmsle,
    },
    "inference": "mean of five restored-checkpoint log predictions, then expm1",
    "duration_seconds": time.perf_counter() - started_time,
    "artifacts": {
        "notebook": "notebooks/feature_engineering/basemlp_outlier_1299_submission.ipynb",
        "checkpoint_dir": str(checkpoint_dir.relative_to(ROOT)),
        "preprocessor_dir": str(preprocessor_dir.relative_to(ROOT)),
        "epoch_log_dir": str(epoch_log_dir.relative_to(ROOT)),
        "oof_predictions": str(NOTEBOOK_OOF_PATH.relative_to(ROOT)),
        "test_fold_predictions": str(NOTEBOOK_FOLD_PREDICTIONS_PATH.relative_to(ROOT)),
        "submission": str(NOTEBOOK_SUBMISSION_PATH.relative_to(ROOT)),
        "submission_validation": str(SUBMISSION_VALIDATION_PATH.relative_to(ROOT)),
    },
    "submission_validation": submission_validation,
    "kaggle_scores": {"public": "unverified", "private": "unverified"},
}
METRICS_PATH.write_text(json.dumps(metrics, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
SUBMISSION_VALIDATION_PATH.write_text(
    json.dumps(submission_validation, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
display(pd.Series(checks, name="Reproducibility and submission checks"))
assert submission_validation["all_checks_passed"], submission_validation

columns                            True
rows                               True
dtypes                             True
id_order                           True
unique_ids                         True
missing_predictions                True
finite_predictions                 True
positive_predictions               True
fold_scores_exact                  True
oof_rmsle_exact                    True
checkpoint_weights_exact_5_of_5    True
fold_artifacts                     True
Name: Reproducibility and submission checks, dtype: bool

## Takeaways

- 6개 code cell이 top-to-bottom 실행됐고 error output은 0개다.
- 원본 BaseMLP와 모델·전처리·fold·seed·학습 recipe가 같고, 학습 행에서 `Id=1299`만 제외했다.
- fold 점수와 OOF RMSLE, 5개 checkpoint 가중치가 이전 독립 `1299` 민감도 실행과 정확히 일치했다.
- submission `submissions/submission_BASEMLP-20260719-OUTLIER-1299-NOTEBOOK-SUB-01.csv`은 `Id,SalePrice`, 1,459행, test Id 순서, 유한·양수 예측을 통과했다.
- 실제 Kaggle 점수는 제출 후 별도로 기록해야 한다.